# Lecture 4: Exploratory Data Analysis
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Conduct a structured EDA on a real dataset
2. Handle missing values and outliers
3. Analyse distributions and relationships
4. Generate insights to guide modelling decisions

In [ ]:
# Install if running on Colab: !pip install -q pandas matplotlib seaborn scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
print('Ready.')

## 4.1 Loading a Real Dataset

We use the Student Performance dataset from UCI ML Repository.

In [ ]:
# Download UCI Student Performance dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip'
import urllib.request, zipfile, io
try:
    response = urllib.request.urlopen(url, timeout=10)
    zf = zipfile.ZipFile(io.BytesIO(response.read()))
    df = pd.read_csv(zf.open('student-mat.csv'), sep=';')
    print('Dataset downloaded successfully.')
except Exception:
    print('Download failed. Generating synthetic equivalent...')
    np.random.seed(42)
    n = 395
    df = pd.DataFrame({
        'age': np.random.randint(15, 22, n),
        'studytime': np.random.randint(1, 5, n),
        'absences': np.random.poisson(4, n),
        'G1': np.random.randint(5, 19, n),
        'G2': np.random.randint(5, 19, n),
        'G3': np.random.randint(0, 20, n),
        'sex': np.random.choice(['M', 'F'], n),
        'address': np.random.choice(['U', 'R'], n),
        'Medu': np.random.randint(0, 5, n),
        'failures': np.random.randint(0, 4, n)
    })
    # Introduce realistic missing values
    for col in ['G1', 'absences']:
        df.loc[df.sample(frac=0.05).index, col] = np.nan
    print('Synthetic dataset created.')

print(f'Shape: {df.shape}')

In [ ]:
# Step 1: Basic structure
print('=== Dataset Info ===')
df.info()
print('\n=== First 5 rows ===')
df.head()

In [ ]:
# Step 2: Summary statistics
df.describe()

In [ ]:
# Step 3: Missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0])

## 4.2 Handling Missing Values

In [ ]:
# Impute numerical columns with median
num_cols = df.select_dtypes(include=np.number).columns
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])
print('After imputation:')
print(df.isnull().sum().sum(), 'missing values remaining')

## 4.3 Distribution Analysis

In [ ]:
# Target variable: G3 (final grade)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['G3'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Final Grade Distribution')
axes[0].set_xlabel('G3 (Final Grade)')

# Box plot
df.boxplot(column='G3', ax=axes[1])
axes[1].set_title('G3 Box Plot')

# Study time
df['studytime'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Study Time Distribution')
axes[2].set_xlabel('Study Time (1=low, 4=high)')

plt.tight_layout()
plt.show()

## 4.4 Outlier Detection (IQR Method)

In [ ]:
col = 'absences'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df[col] < lower) | (df[col] > upper)]
print(f'Outlier bounds for "{col}": lower={lower:.1f}, upper={upper:.1f}')
print(f'Number of outliers: {len(outliers)}')
print(f'Max absences in dataset: {df[col].max():.0f}')

## 4.5 Relationship Analysis

In [ ]:
num_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Student Performance Features')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix for grade variables
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='studytime', y='G3', alpha=0.5, ax=axes[0])
axes[0].set_title('Study Time vs Final Grade')
if 'sex' in df.columns:
    sns.boxplot(data=df, x='sex', y='G3', ax=axes[1], palette='Set2')
    axes[1].set_title('Final Grade by Sex')
plt.tight_layout()
plt.show()

### Exercise

Using the student performance DataFrame `df`:

1. Create a new binary column `passed` where G3 >= 10.
2. Calculate the pass rate overall and by sex.
3. Plot a bar chart showing mean G3 by study time level.
4. Identify the top 3 features most correlated with G3 using the correlation matrix.

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*